In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph , START , END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
class SubState(TypedDict):
    input_text : str
    translated_text : str

In [ ]:
subgraph_llm = ChatOpenAI(model = 'gpt-4o')

In [ ]:
def translated_text(state: SubState):
    prompt  = f"""
    Translate the following text to Hindi.
    Keep it natural and clear .Do not add extra content.
    Text:
    {state["input_text"]}
    """.strip()
    translate_text = subgraph_llm.invoke(prompt)
    return {'translate_text': translate_text }

In [ ]:
subgraph_builder  = StateGraph(SubState)


subgraph_builder.add_node('translate_text' , translate_text)

subgraph_builder.add_edge(START , 'translate_text')
subgraph_builder.add_edge('translate_text', END)

subgraph = subgraph_builder.compile()

In [ ]:
class ParentState(TypedDict):
    question : str
    answer_end : str
    answer_urdu : str

In [ ]:
parent_llm = ChatOpenAI(model = 'gpt-4o')

In [ ]:
def generate_answer(state : ParentState):
    answer = parent_llm.invoke(f"You are a  helpful assistant . Answer clearly.\n\nQuestions : {state['question']}").content
    return {'answer_end': answer}

In [ ]:
def translate_answer(state: ParentState):
    
    result = subgraph.invoke({'input_text':state['answer_end']})
    
    return {'answer_urdu' : result['translated_text']}

In [ ]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("answer" , generate_answer)
parent_builder.add_node("translate":translate_answer)

parent_builder.add_edge(START , "answer")
parent_builder.add_edge("answer" , "translate")
parent_builder.add_edge("translate" , END)

In [ ]:
graph = parent_builder.compile()
graph 

In [ ]:
graph.invoke({'question':'what is quantum physics'})